# Programming Assignment: Building Support Vector Machines from Scratch

Welcome to the assignment for Support Vector Machine (SVM) classification.

Support Vector Machines are among the most powerful and versatile supervised learning algorithms. While neural networks have dominated recent headlines, SVMs remain the go-to choice for many practical applications due to their strong theoretical foundations, excellent performance on medium-sized datasets, and ability to work effectively in high-dimensional spaces.

In this assignment, you'll implement SVM classifiers using both scikit-learn and from-scratch implementations using NumPy. You'll work with linear and non-linear decision boundaries, explore the impact of hyperparameters, and understand why SVMs are particularly effective for complex classification tasks.

SVMs work by finding the optimal hyperplane that maximizes the margin between classes. The "support vectors" are the critical data points closest to the decision boundary that define this margin. This geometric approach makes SVMs naturally robust and often requires fewer training samples than other algorithms.

**What You Will Do in This Assignment**

* Understand the mathematical foundation of SVMs including margin maximization and the kernel trick.
* Implement Linear SVM for linearly separable data and visualize support vectors and margins.
* Demonstrate the critical importance of feature scaling in SVM performance.
* Apply kernel methods (RBF) to handle non-linearly separable data.
* Compare different kernels (linear, polynomial, RBF) and understand when each excels.
* Implement multi-class classification strategies (One-vs-Rest and One-vs-One).
* Build a complete Linear SVM from scratch using gradient descent.
* Tune hyperparameters (C, gamma) and understand their impact on model behavior.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#imports)
- [1 - Introduction to Support Vector Machines](#1)
    - [1.1 - The Maximum Margin Principle](#1-1)
    - [1.2 - Support Vectors and the Margin](#1-2)
    - [1.3 - The Kernel Trick](#1-3)
- [2 - Linear SVM for Separable Data](#2)
    - [2.1 - Preparing the Dataset](#2-1)
    - **[Exercise 1 - train_linear_svm](#ex-1)**
    - [2.2 - Visualizing Decision Boundaries and Support Vectors](#2-2)
- [3 - Feature Scaling Impact](#3)
    - **[Exercise 2 - compare_scaled_unscaled_svm](#ex-2)**
- [4 - Kernel SVM with RBF](#4)
    - [4.1 - Understanding the RBF Kernel](#4-1)
    - **[Exercise 3 - tune_rbf_svm](#ex-3)**
- [5 - Kernel Selection and Comparison](#5)
    - **[Exercise 4 - compare_kernels](#ex-4)**
- [6 - Multi-Class Classification](#6)
    - [6.1 - One-vs-Rest vs One-vs-One](#6-1)
    - **[Exercise 5 - train_multiclass_svm_strategies](#ex-5)**
- [7 - Linear SVM from Scratch](#7)
    - [7.1 - The Primal Formulation](#7-1)
    - **[Exercise 6a - hinge_loss](#ex-6a)**
    - **[Exercise 6b - svm_primal_gradients](#ex-6b)**
    - **[Exercise 6c - train_linear_svm_from_scratch](#ex-6c)**

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import make_blobs, make_moons, make_circles, load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

<a name='1'></a>
## 1 - Introduction to Support Vector Machines

Support Vector Machines (SVMs) are powerful supervised learning algorithms that excel at classification tasks. Unlike algorithms that simply try to separate classes, SVMs aim to find the **optimal separating hyperplane** that maximizes the distance (margin) between classes.

<a name='1-1'></a>
### 1.1 - The Maximum Margin Principle

The key idea behind SVMs is **margin maximization**. Given a linearly separable dataset, there are infinitely many hyperplanes that can separate the classes. SVM chooses the one that maximizes the minimum distance to any training point.

For a linear classifier, the decision function is:

$$f(x) = w^T x + b$$

where $w$ is the weight vector (perpendicular to the hyperplane) and $b$ is the bias term.

The **margin** is defined as:

$$\text{margin} = \frac{2}{\|w\|}$$

To maximize the margin, we minimize $\|w\|^2$, subject to the constraint that all points are correctly classified:

$$y_i(w^T x_i + b) \geq 1, \quad \forall i$$

This leads to the **hard-margin SVM optimization problem**:

$$\min_{w,b} \frac{1}{2}\|w\|^2$$
$$\text{subject to: } y_i(w^T x_i + b) \geq 1, \quad \forall i$$

<a name='1-2'></a>
### 1.2 - Support Vectors and the Margin

**Support vectors** are the training samples that lie exactly on the margin boundaries. These are the critical points that define the decision boundary. Remarkably, the optimal hyperplane depends only on these support vectors—other training points could be removed without changing the solution!

For real-world data that isn't perfectly separable, we use **soft-margin SVM**, which allows some misclassifications by introducing slack variables $\xi_i$:

$$\min_{w,b,\xi} \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{m}\xi_i$$
$$\text{subject to: } y_i(w^T x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

The parameter $C$ controls the trade-off between:
- **Large C**: Smaller margin, fewer misclassifications (risk of overfitting)
- **Small C**: Larger margin, more misclassifications (risk of underfitting)

<a name='1-3'></a>
### 1.3 - The Kernel Trick

For non-linearly separable data, we can transform features into a higher-dimensional space where they become separable. The **kernel trick** allows us to compute dot products in this high-dimensional space without explicitly computing the transformation!

Common kernels include:

**Linear Kernel**: $K(x_i, x_j) = x_i^T x_j$

**Polynomial Kernel**: $K(x_i, x_j) = (x_i^T x_j + c)^d$

**RBF (Radial Basis Function) Kernel**: $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$

The RBF kernel is particularly powerful—it can approximate any continuous function and works well when there's no prior knowledge about the data distribution.

<a name='helpers'></a>
## Helper Functions

We'll define a utility function for visualizing 2D decision boundaries.

In [ ]:
def plot_decision_boundary_2d(model, X, y, title="Decision Boundary", h=0.02):
    """Plot a 2D decision boundary."""
    X = np.asarray(X)
    y = np.asarray(y)

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(9, 7))
    plt.contourf(xx, yy, Z, alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", s=35)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

<a name='2'></a>
## 2 - Linear SVM for Separable Data

Let's start by implementing a linear SVM on a dataset that is approximately linearly separable. This will help us understand the basic concepts before moving to more complex scenarios.

<a name='2-1'></a>
### 2.1 - Preparing the Dataset

We'll create a synthetic dataset with two well-separated clusters using scikit-learn's `make_blobs` function.

In [ ]:
# Create a linearly-separable dataset
X_lin, y_lin = make_blobs(n_samples=400, centers=2, cluster_std=1.2, random_state=42)
y_lin = y_lin.astype(int)

plt.figure(figsize=(7,6))
plt.scatter(X_lin[:,0], X_lin[:,1], c=y_lin, edgecolors="k", s=30)
plt.title("Blobs Dataset (Approximately Linearly Separable)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_lin, y_lin, test_size=0.25, random_state=42, stratify=y_lin
)

print(f"Training samples: {X_train_lin.shape[0]}")
print(f"Testing samples: {X_test_lin.shape[0]}")
print(f"Number of features: {X_train_lin.shape[1]}")

<a name='ex-1'></a>
#### **Exercise 1 - `train_linear_svm`**

**Your Task:**

Implement the `train_linear_svm` function that trains a linear Support Vector Machine classifier using scikit-learn's `SVC` class.

You need to implement:

* **Create an SVC instance with linear kernel**:
    * Use `SVC(kernel='linear', C=C)` to create the model.
    * The `C` parameter controls the regularization strength (inverse of regularization).

* **Fit the model on training data**:
    * Use the `.fit()` method to train on the provided features and labels.

* **Return the trained model**:
    * Return the fitted SVC object for later prediction and analysis.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For creating the model:**
* Import not needed (already imported at top): `from sklearn.svm import SVC`.
* Create model: `model = SVC(kernel='linear', C=C)`.

**For training:**
* Fit the model: `model.fit(X, y)`.
* The fit method modifies the model in-place and also returns it.

**For returning:**
* Simply return the fitted model object.

</details>

In [ ]:
# GRADED FUNCTION: train_linear_svm
def train_linear_svm(X, y, C=1.0):
    """
    Train a linear SVM classifier.

    Args:
        X: Feature array of shape (m, n_features)
        y: Target array of shape (m,)
        C: Regularization parameter (default: 1.0)

    Returns:
        model: Trained SVC with linear kernel
    """
    ### START CODE HERE ###
    
    # Create SVC with linear kernel
    
    # Fit the model on training data
    
    ### END CODE HERE ###
    
    return model

In [ ]:
# Test your implementation
model_lin = train_linear_svm(X_train_lin, y_train_lin, C=1.0)
y_pred = model_lin.predict(X_test_lin)

print("Test Accuracy:", accuracy_score(y_test_lin, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_lin, y_pred))

print(f"\nNumber of support vectors: {len(model_lin.support_vectors_)}")
print(f"Support vectors per class: {model_lin.n_support_}")

##### **Expected Output**
```
Test Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        50

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

Number of support vectors: 6
Support vectors per class: [3 3]
```

In [ ]:
unittests.exercise_1(train_linear_svm)

<a name='2-2'></a>
### 2.2 - Visualizing Decision Boundaries and Support Vectors

One of the most appealing aspects of SVMs is their interpretability. We can visualize the decision boundary and identify which training points are support vectors—the critical points that define the margin.

In [ ]:
# Visualize decision boundary
plot_decision_boundary_2d(model_lin, X_train_lin, y_train_lin, title="Linear SVM Decision Boundary")

# Highlight support vectors
sv = model_lin.support_vectors_
plt.figure(figsize=(9,7))
plt.scatter(X_train_lin[:,0], X_train_lin[:,1], c=y_train_lin, edgecolors="k", s=35, alpha=0.6)
plt.scatter(sv[:,0], sv[:,1], s=180, facecolors="none", edgecolors="red", linewidths=2.5, label="Support Vectors")
plt.title("Support Vectors Highlighted")
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend()
plt.show()

print(f"Only {len(sv)} out of {len(X_train_lin)} training points are support vectors!")
print("These critical points alone define the entire decision boundary.")

<a name='3'></a>
## 3 - Feature Scaling Impact

SVMs are highly sensitive to feature scales because they rely on distance calculations. Features with larger scales will dominate the distance computation, leading to suboptimal decision boundaries. Let's demonstrate this critical issue.

<a name='ex-2'></a>
#### **Exercise 2 - `compare_scaled_unscaled_svm`**

**Your Task:**

Implement the `compare_scaled_unscaled_svm` function that trains RBF SVM on both unscaled and scaled features to demonstrate the importance of feature scaling.

You need to implement:

* **Train on unscaled data**:
    * Create and fit an SVC with RBF kernel on the raw training data.
    * Predict on unscaled test data and calculate accuracy.

* **Scale the data**:
    * Use `StandardScaler()` to normalize features (zero mean, unit variance).
    * Fit the scaler on training data only, then transform both train and test sets.

* **Train on scaled data**:
    * Create and fit an SVC with RBF kernel on scaled training data.
    * Predict on scaled test data and calculate accuracy.

* **Return results**:
    * Return a dictionary with both accuracies, both models, and the scaler.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For unscaled model:**
* Create: `unscaled_model = SVC(kernel='rbf', C=C, gamma=gamma)`.
* Fit: `unscaled_model.fit(X_train, y_train)`.
* Predict and score: `unscaled_acc = accuracy_score(y_test, unscaled_model.predict(X_test))`.

**For scaling:**
* Create scaler: `scaler = StandardScaler()`.
* Fit and transform train: `X_train_scaled = scaler.fit_transform(X_train)`.
* Transform test only: `X_test_scaled = scaler.transform(X_test)`.

**For scaled model:**
* Same as unscaled but use scaled data.

**Return dictionary:**
* `results = {'unscaled_acc': ..., 'scaled_acc': ..., 'unscaled_model': ..., 'scaled_model': ..., 'scaler': ...}`.

</details>

In [ ]:
# Create a dataset with feature scale imbalance
X_scale, y_scale = make_moons(n_samples=400, noise=0.25, random_state=42)
y_scale = y_scale.astype(int)

# Artificially create scale imbalance (multiply one feature by 25)
X_imbalanced = X_scale.copy()
X_imbalanced[:, 1] *= 25.0

X_tr, X_te, y_tr, y_te = train_test_split(
    X_imbalanced, y_scale, test_size=0.25, random_state=42, stratify=y_scale
)

print(f"Feature 1 range: [{X_tr[:, 0].min():.2f}, {X_tr[:, 0].max():.2f}]")
print(f"Feature 2 range: [{X_tr[:, 1].min():.2f}, {X_tr[:, 1].max():.2f}]")
print("\nNotice the huge scale difference!")

In [ ]:
# GRADED FUNCTION: compare_scaled_unscaled_svm
def compare_scaled_unscaled_svm(X_train, X_test, y_train, y_test, C=1.0, gamma='scale'):
    """
    Train RBF SVM on unscaled vs scaled features and compare performance.

    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training labels
        y_test: Test labels
        C: Regularization parameter
        gamma: Kernel coefficient

    Returns:
        results: dict with keys:
            'unscaled_acc': float - Accuracy on unscaled data
            'scaled_acc': float - Accuracy on scaled data
            'unscaled_model': trained SVC on unscaled data
            'scaled_model': trained SVC on scaled data
            'scaler': fitted StandardScaler
    """
    results = {}
    
    ### START CODE HERE ###
    
    # Train on unscaled data
    
    # Scale the data
    
    # Train on scaled data
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Test your implementation
results_scale = compare_scaled_unscaled_svm(X_tr, X_te, y_tr, y_te, C=1.0, gamma='scale')

print("="*60)
print("FEATURE SCALING IMPACT")
print("="*60)
print(f"Unscaled accuracy: {results_scale['unscaled_acc']:.4f}")
print(f"Scaled accuracy:   {results_scale['scaled_acc']:.4f}")
print(f"\nImprovement: {(results_scale['scaled_acc'] - results_scale['unscaled_acc'])*100:.1f}%")
print("="*60)

##### **Expected Output**
```
============================================================
FEATURE SCALING IMPACT
============================================================
Unscaled accuracy: 0.5000
Scaled accuracy:   0.9200

Improvement: 42.0%
============================================================
```

In [ ]:
unittests.exercise_2(compare_scaled_unscaled_svm)

In [ ]:
# Visualize the difference
plot_decision_boundary_2d(results_scale["unscaled_model"], X_tr, y_tr, title="RBF SVM (Unscaled Data)")

X_tr_scaled = results_scale["scaler"].transform(X_tr)
plot_decision_boundary_2d(results_scale["scaled_model"], X_tr_scaled, y_tr, title="RBF SVM (Scaled Data)")

<a name='4'></a>
## 4 - Kernel SVM with RBF

For data that isn't linearly separable, we need non-linear decision boundaries. The RBF (Radial Basis Function) kernel is one of the most popular choices for this task.

<a name='4-1'></a>
### 4.1 - Understanding the RBF Kernel

The RBF kernel is defined as:

$$K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$$

The parameter $\gamma$ controls the influence of individual training samples:
- **Large $\gamma$**: Each training point has a small radius of influence → more complex boundaries (risk of overfitting)
- **Small $\gamma$**: Each training point has a large radius of influence → smoother boundaries (risk of underfitting)

Combined with the regularization parameter $C$, we have two hyperparameters to tune for optimal performance.

In [ ]:
# Create a non-linearly separable dataset (moons)
X_moons, y_moons = make_moons(n_samples=400, noise=0.2, random_state=42)
y_moons = y_moons.astype(int)

plt.figure(figsize=(7,6))
plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, edgecolors="k", s=30)
plt.title("Moons Dataset (Non-linearly Separable)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()

print("This dataset cannot be separated by a straight line!")

<a name='ex-3'></a>
#### **Exercise 3 - `tune_rbf_svm`**

**Your Task:**

Implement the `tune_rbf_svm` function that uses GridSearchCV to find the best hyperparameters (C and gamma) for an RBF SVM.

You need to implement:

* **Set up parameter grid**:
    * If not provided, use reasonable default ranges for C and gamma.
    * Example: `C_values = [0.1, 1, 10, 100]` and `gamma_values = [0.001, 0.01, 0.1, 1]`.

* **Create GridSearchCV**:
    * Use `GridSearchCV` with an SVC(kernel='rbf').
    * Use 5-fold cross-validation to evaluate each parameter combination.

* **Fit and extract results**:
    * Fit the grid search on the training data.
    * Extract the best model, best parameters, and best cross-validation score.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For parameter grid:**
* Check if provided: `if C_values is None: C_values = [0.1, 1, 10, 100]`.
* Create dict: `param_grid = {'C': C_values, 'gamma': gamma_values}`.

**For GridSearchCV:**
* Create base model: `svc = SVC(kernel='rbf')`.
* Create grid: `grid = GridSearchCV(svc, param_grid, cv=5, scoring='accuracy')`.
* Fit: `grid.fit(X, y)`.

**For extracting results:**
* Best model: `best_model = grid.best_estimator_`.
* Best params: `best_params = grid.best_params_`.
* Best score: `best_score = grid.best_score_`.

</details>

In [ ]:
# GRADED FUNCTION: tune_rbf_svm
def tune_rbf_svm(X, y, C_values=None, gamma_values=None):
    """
    Tune RBF SVM hyperparameters using GridSearchCV.

    Args:
        X: Feature array
        y: Target array
        C_values: list of C values to try (optional)
        gamma_values: list of gamma values to try (optional)

    Returns:
        best_model: trained SVC with RBF kernel using best parameters
        best_params: dict of best parameters found
        best_score: float - best cross-validation score
    """
    ### START CODE HERE ###
    
    # Set default parameter values if not provided
    
    # Create parameter grid
    
    # Create and fit GridSearchCV
    
    # Extract best results
    
    ### END CODE HERE ###
    
    return best_model, best_params, best_score

In [ ]:
# Test your implementation
best_rbf, best_params, best_cv = tune_rbf_svm(X_moons, y_moons)

print("="*60)
print("RBF SVM HYPERPARAMETER TUNING")
print("="*60)
print(f"Best parameters: {best_params}")
print(f"Best CV score: {best_cv:.4f}")
print("="*60)

# Visualize the result
plot_decision_boundary_2d(best_rbf, X_moons, y_moons, title="Best RBF SVM (Tuned Hyperparameters)")

# Highlight support vectors
sv = best_rbf.support_vectors_
plt.figure(figsize=(9,7))
plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, edgecolors="k", s=35, alpha=0.6)
plt.scatter(sv[:,0], sv[:,1], s=180, facecolors="none", edgecolors="red", linewidths=2.5, label="Support Vectors")
plt.title("RBF SVM Support Vectors (Non-linear Boundary)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend()
plt.show()

##### **Expected Output**
```
============================================================
RBF SVM HYPERPARAMETER TUNING
============================================================
Best parameters: {'C': X, 'gamma': X.XXX}
Best CV score: 0.9XXX
============================================================
```

In [ ]:
unittests.exercise_3(tune_rbf_svm)

<a name='5'></a>
## 5 - Kernel Selection and Comparison

Different datasets have different characteristics, and no single kernel works best for all problems. Let's compare three popular kernels across multiple datasets to understand their strengths and weaknesses.

<a name='ex-4'></a>
#### **Exercise 4 - `compare_kernels`**

**Your Task:**

Implement the `compare_kernels` function that compares linear, polynomial, and RBF kernels across multiple datasets.

You need to implement:

* **Loop through datasets**:
    * For each dataset in the dictionary, extract features and labels.
    * Split into train/test sets for proper evaluation.

* **Train three kernel types**:
    * Linear kernel: `SVC(kernel='linear', C=C)`.
    * Polynomial kernel: `SVC(kernel='poly', C=C, degree=degree, gamma=gamma)`.
    * RBF kernel: `SVC(kernel='rbf', C=C, gamma=gamma)`.

* **Evaluate and store results**:
    * Calculate test accuracy for each kernel on each dataset.
    * Store results in a pandas DataFrame with datasets as rows and kernels as columns.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For dataset loop:**
* Iterate: `for name, (X, y) in datasets.items()`.
* Split: `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)`.

**For each kernel:**
* Create, fit, predict: 
  ```python
  model = SVC(kernel='linear', C=C)
  model.fit(X_train, y_train)
  acc = accuracy_score(y_test, model.predict(X_test))
  ```

**For storing results:**
* Use dictionary: `results = {dataset_name: {kernel_name: accuracy}}`.
* Convert to DataFrame: `results_df = pd.DataFrame(results).T`.

</details>

In [ ]:
# Create three different datasets
X_circles, y_circles = make_circles(n_samples=400, noise=0.12, factor=0.5, random_state=42)
y_circles = y_circles.astype(int)

datasets = {
    "blobs": (X_lin, y_lin),
    "moons": (X_moons, y_moons),
    "circles": (X_circles, y_circles),
}

# Visualize all datasets
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, (X, y)) in enumerate(datasets.items()):
    axes[idx].scatter(X[:,0], X[:,1], c=y, edgecolors="k", s=30)
    axes[idx].set_title(f"{name.capitalize()} Dataset")
    axes[idx].set_xlabel("x1")
    axes[idx].set_ylabel("x2")
plt.tight_layout()
plt.show()

In [ ]:
# GRADED FUNCTION: compare_kernels
def compare_kernels(datasets, C=1.0, degree=3, gamma='scale'):
    """
    Compare linear, polynomial, and RBF kernels across multiple datasets.

    Args:
        datasets: dict mapping dataset name -> (X, y) tuple
        C: Regularization parameter
        degree: Degree for polynomial kernel
        gamma: Kernel coefficient for poly and RBF

    Returns:
        results_df: pandas DataFrame with datasets as rows and kernels as columns
                   Each cell contains the test accuracy
    """
    ### START CODE HERE ###
    
    # Initialize results dictionary
    
    # Loop through each dataset
    
        # Split into train/test
        
        # Train and evaluate linear kernel
        
        # Train and evaluate polynomial kernel
        
        # Train and evaluate RBF kernel
        
    # Convert results to DataFrame
    
    ### END CODE HERE ###
    
    return results_df

In [ ]:
# Test your implementation
results_df = compare_kernels(datasets, C=1.0, degree=3, gamma='scale')

print("="*60)
print("KERNEL COMPARISON ACROSS DATASETS")
print("="*60)
print(results_df)
print("="*60)
print("\nKey Insights:")
print("• Linear kernel excels on linearly separable data (blobs)")
print("• RBF kernel is versatile and performs well on all datasets")
print("• Polynomial kernel can struggle without proper tuning")

##### **Expected Output**
```
============================================================
KERNEL COMPARISON ACROSS DATASETS
============================================================
          linear  polynomial       rbf
blobs     1.0000      0.XXXX    1.0000
moons     0.8XXX      0.XXXX    0.9XXX
circles   0.5XXX      0.XXXX    0.9XXX
============================================================
```

In [ ]:
unittests.exercise_4(compare_kernels)

<a name='6'></a>
## 6 - Multi-Class Classification

SVMs are inherently binary classifiers. For multi-class problems, we need strategies to combine multiple binary classifiers. The two most common approaches are One-vs-Rest (OvR) and One-vs-One (OvO).

<a name='6-1'></a>
### 6.1 - One-vs-Rest vs One-vs-One

**One-vs-Rest (OvR)**:
- Train $K$ binary classifiers (one for each class)
- Each classifier separates one class from all others
- For prediction, choose the class with the highest confidence score
- Training time: $O(K \cdot n)$ where $K$ is number of classes

**One-vs-One (OvO)**:
- Train $\frac{K(K-1)}{2}$ binary classifiers (one for each pair of classes)
- Each classifier separates two specific classes
- For prediction, use voting: each classifier votes for one class
- Training time: $O(K^2 \cdot n/K^2) = O(n)$ but with more models

scikit-learn's SVC uses OvO by default for multi-class problems.

In [ ]:
# Load digits dataset (10 classes: 0-9)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Number of samples: {X_digits.shape[0]}")
print(f"Number of features: {X_digits.shape[1]}")
print(f"Number of classes: {len(np.unique(y_digits))}")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for idx, ax in enumerate(axes.flat):
    ax.imshow(digits.images[idx], cmap='gray')
    ax.set_title(f"Label: {digits.target[idx]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

# Split and scale
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_digits, y_digits, test_size=0.25, random_state=42, stratify=y_digits
)

scaler_d = StandardScaler()
X_tr_d_s = scaler_d.fit_transform(X_tr_d)
X_te_d_s = scaler_d.transform(X_te_d)

<a name='ex-5'></a>
#### **Exercise 5 - `train_multiclass_svm_strategies`**

**Your Task:**

Implement the `train_multiclass_svm_strategies` function that trains both One-vs-Rest and One-vs-One strategies and compares their performance.

You need to implement:

* **Train One-vs-Rest (OvR)**:
    * Use `OneVsRestClassifier` wrapper around an SVC.
    * Fit on training data and predict on test data.
    * Calculate test accuracy.

* **Train One-vs-One (OvO)**:
    * Use `OneVsOneClassifier` wrapper around an SVC.
    * Fit on training data and predict on test data.
    * Calculate test accuracy.

* **Return results**:
    * Return dictionary with both accuracies and both trained models.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For One-vs-Rest:**
* Create base estimator: `svc = SVC(kernel=kernel, C=C, gamma=gamma)`.
* Wrap: `ovr_model = OneVsRestClassifier(svc)`.
* Fit: `ovr_model.fit(X_train, y_train)`.
* Evaluate: `ovr_acc = accuracy_score(y_test, ovr_model.predict(X_test))`.

**For One-vs-One:**
* Similar to OvR but use: `ovo_model = OneVsOneClassifier(svc)`.

**Return dictionary:**
* `results = {'ovr_acc': ..., 'ovo_acc': ..., 'ovr_model': ..., 'ovo_model': ...}`.

</details>

In [ ]:
# GRADED FUNCTION: train_multiclass_svm_strategies
def train_multiclass_svm_strategies(X_train, y_train, X_test, y_test, C=2.0, kernel='rbf', gamma='scale'):
    """
    Train OvR and OvO SVM strategies for multi-class classification.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        C: Regularization parameter
        kernel: Kernel type
        gamma: Kernel coefficient

    Returns:
        results: dict with keys:
            'ovr_acc': float - One-vs-Rest test accuracy
            'ovo_acc': float - One-vs-One test accuracy
            'ovr_model': trained OneVsRestClassifier
            'ovo_model': trained OneVsOneClassifier
    """
    results = {}
    
    ### START CODE HERE ###
    
    # Create base SVC
    
    # Train One-vs-Rest
    
    # Train One-vs-One
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Test your implementation
multi_results = train_multiclass_svm_strategies(X_tr_d_s, y_tr_d, X_te_d_s, y_te_d, C=2.0, kernel='rbf', gamma='scale')

print("="*60)
print("MULTI-CLASS SVM STRATEGIES")
print("="*60)
print(f"One-vs-Rest (OvR) accuracy: {multi_results['ovr_acc']:.4f}")
print(f"One-vs-One (OvO) accuracy:  {multi_results['ovo_acc']:.4f}")
print("="*60)

# Number of classifiers trained
n_classes = len(np.unique(y_tr_d))
n_ovr = n_classes
n_ovo = n_classes * (n_classes - 1) // 2

print(f"\nNumber of binary classifiers:")
print(f"  OvR: {n_ovr} classifiers (one per class)")
print(f"  OvO: {n_ovo} classifiers (one per class pair)")

##### **Expected Output**
```
============================================================
MULTI-CLASS SVM STRATEGIES
============================================================
One-vs-Rest (OvR) accuracy: 0.9XXX
One-vs-One (OvO) accuracy:  0.9XXX
============================================================

Number of binary classifiers:
  OvR: 10 classifiers (one per class)
  OvO: 45 classifiers (one per class pair)
```

In [ ]:
unittests.exercise_5(train_multiclass_svm_strategies)

<a name='7'></a>
## 7 - Linear SVM from Scratch

Now that you understand how SVMs work, let's implement a linear SVM from scratch using gradient descent. This will deepen your understanding of the optimization process behind SVMs.

<a name='7-1'></a>
### 7.1 - The Primal Formulation

We'll solve the soft-margin SVM primal problem:

$$\min_{w,b} \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{m}\max(0, 1 - y_i(w^T x_i + b))$$

The second term is the **hinge loss**: $\ell(y, s) = \max(0, 1 - ys)$ where $s = w^T x + b$ is the decision score.

We'll use **subgradient descent** because the hinge loss is not differentiable at $ys = 1$. The subgradient is:

For the regularization term: $\nabla_w \frac{1}{2}\|w\|^2 = w$

For the hinge loss term:
$$\nabla_w \ell = \begin{cases} -y_i x_i & \text{if } y_i(w^T x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

$$\nabla_b \ell = \begin{cases} -y_i & \text{if } y_i(w^T x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

**Note**: We'll work with labels in $\{-1, +1\}$ for mathematical convenience.

<a name='ex-6a'></a>
#### **Exercise 6a - `hinge_loss`**

**Your Task:**

Implement the `hinge_loss` function that computes the hinge loss for SVM.

You need to implement:

* **Calculate hinge loss for each sample**:
    * For each sample, compute: $\max(0, 1 - y_i \cdot s_i)$
    * where $y_i \in \{-1, +1\}$ and $s_i$ is the decision score.

* **Return mean loss**:
    * Average the losses across all samples.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For hinge loss:**
* Compute margin violation: `margin = 1 - y_true * scores`.
* Apply max: `loss = np.maximum(0, margin)`.
* Return mean: `return np.mean(loss)`.

**Alternative vectorized form:**
* One line: `return np.mean(np.maximum(0, 1 - y_true * scores))`.

</details>

In [ ]:
# GRADED FUNCTION: hinge_loss
def hinge_loss(y_true, scores):
    """
    Compute hinge loss: mean(max(0, 1 - y*s))

    Args:
        y_true: numpy array of shape (m,) - labels in {-1, +1}
        scores: numpy array of shape (m,) - raw decision scores (w^T x + b)

    Returns:
        loss: float - mean hinge loss
    """
    ### START CODE HERE ###
    
    # Calculate hinge loss for each sample
    
    # Return mean loss
    
    ### END CODE HERE ###
    
    return loss

In [ ]:
# Test hinge_loss
y_test = np.array([-1, 1, 1, -1, 1])
scores_test = np.array([-2.0, 1.5, 0.3, 1.0, -0.5])

loss = hinge_loss(y_test, scores_test)
print(f"Hinge loss: {loss:.4f}")

# Test perfect predictions
scores_perfect = np.array([-2.0, 2.0, 2.0, -2.0, 2.0])
loss_perfect = hinge_loss(y_test, scores_perfect)
print(f"Hinge loss (perfect predictions): {loss_perfect:.4f}")

##### **Expected Output**
```
Hinge loss: 0.6600
Hinge loss (perfect predictions): 0.0000
```

In [ ]:
unittests.exercise_6a(hinge_loss)

<a name='ex-6b'></a>
#### **Exercise 6b - `svm_primal_gradients`**

**Your Task:**

Implement the `svm_primal_gradients` function that computes subgradients of the SVM primal objective.

You need to implement:

* **Compute decision scores**:
    * Calculate $s_i = w^T x_i + b$ for all samples.

* **Identify margin violations**:
    * Find samples where $y_i \cdot s_i < 1$ (these contribute to the gradient).

* **Compute gradient for w**:
    * Start with regularization term: `grad_w = w`.
    * Add hinge loss gradient: for violated samples, subtract $C \cdot y_i \cdot x_i$.

* **Compute gradient for b**:
    * Sum up $-C \cdot y_i$ for all violated samples.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For scores and violations:**
* Scores: `scores = X @ w + b`.
* Violations mask: `violated = (y * scores) < 1`.

**For grad_w:**
* Start with regularization: `grad_w = w.copy()`.
* Add hinge contribution: `grad_w -= C * (X[violated].T @ y[violated]) / len(y)`.
* Or loop version: 
  ```python
  for i in range(len(y)):
      if violated[i]:
          grad_w -= C * y[i] * X[i] / len(y)
  ```

**For grad_b:**
* `grad_b = -C * np.sum(y[violated]) / len(y)`.

</details>

In [ ]:
# GRADED FUNCTION: svm_primal_gradients
def svm_primal_gradients(X, y, w, b, C):
    """
    Compute subgradients for the SVM primal objective:
        0.5*||w||^2 + C * sum_i max(0, 1 - y_i (w^T x_i + b))

    Args:
        X: numpy array of shape (m, n) - features
        y: numpy array of shape (m,) - labels in {-1, +1}
        w: numpy array of shape (n,) - weight vector
        b: float - bias term
        C: float - regularization parameter

    Returns:
        grad_w: numpy array of shape (n,) - gradient with respect to w
        grad_b: float - gradient with respect to b
    """
    ### START CODE HERE ###
    
    # Compute decision scores
    
    # Identify margin violations
    
    # Compute gradient for w
    
    # Compute gradient for b
    
    ### END CODE HERE ###
    
    return grad_w, grad_b

In [ ]:
# Test svm_primal_gradients
X_test = np.array([[1, 2], [2, 3], [3, 1], [2, 1]])
y_test = np.array([-1, -1, 1, 1])
w_test = np.array([0.5, 0.5])
b_test = 0.0
C_test = 1.0

grad_w, grad_b = svm_primal_gradients(X_test, y_test, w_test, b_test, C_test)
print(f"Gradient w.r.t. w: {grad_w}")
print(f"Gradient w.r.t. b: {grad_b:.4f}")

##### **Expected Output** (values may vary slightly)
```
Gradient w.r.t. w: [X.XXXX X.XXXX]
Gradient w.r.t. b: X.XXXX
```

In [ ]:
unittests.exercise_6b(svm_primal_gradients)

<a name='ex-6c'></a>
#### **Exercise 6c - `train_linear_svm_from_scratch`**

**Your Task:**

Implement the `train_linear_svm_from_scratch` function that trains a linear SVM using gradient descent.

You need to implement:

* **Convert labels to {-1, +1}**:
    * If labels are in {0, 1}, convert them: $y_{new} = 2 \cdot y_{old} - 1$.

* **Initialize parameters**:
    * Initialize `w` to zeros: `np.zeros(n_features)`.
    * Initialize `b` to zero.

* **Gradient descent loop**:
    * For each iteration:
        * Compute gradients using `svm_primal_gradients`.
        * Update parameters: $w \leftarrow w - \alpha \cdot \nabla_w$ and $b \leftarrow b - \alpha \cdot \nabla_b$.
        * Calculate and store objective value: $\frac{1}{2}\|w\|^2 + C \cdot \text{hinge\_loss}$.

* **Return results**:
    * Return trained `w`, `b`, and history of objective values.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For label conversion:**
* Check if binary: `if np.all(np.isin(y, [0, 1]))`.
* Convert: `y_train = 2 * y - 1` (converts 0→-1, 1→+1).

**For initialization:**
* `w = np.zeros(X.shape[1])`.
* `b = 0.0`.

**For training loop:**
```python
for iteration in range(num_iters):
    grad_w, grad_b = svm_primal_gradients(X, y_train, w, b, C)
    w -= lr * grad_w
    b -= lr * grad_b
    
    scores = X @ w + b
    reg_term = 0.5 * np.dot(w, w)
    hinge_term = C * hinge_loss(y_train, scores)
    objective = reg_term + hinge_term
    history.append(objective)
```

</details>

In [ ]:
# GRADED FUNCTION: train_linear_svm_from_scratch
def train_linear_svm_from_scratch(X, y, C=1.0, lr=0.01, num_iters=2000, verbose=True):
    """
    Train a linear SVM using subgradient descent.

    Args:
        X: numpy array of shape (m, n) - training features
        y: numpy array of shape (m,) - labels in {0,1} or {-1,+1}
        C: float - regularization parameter
        lr: float - learning rate
        num_iters: int - number of gradient descent iterations
        verbose: bool - whether to print progress

    Returns:
        w: numpy array of shape (n,) - learned weight vector
        b: float - learned bias term
        history: list - objective value at each iteration
    """
    history = []
    
    ### START CODE HERE ###
    
    # Convert labels to {-1, +1} if necessary
    
    # Initialize parameters
    
    # Gradient descent loop
    
    ### END CODE HERE ###
    
    if verbose and (num_iters >= 100):
        print(f"Iteration {num_iters}: Objective = {history[-1]:.4f}")
    
    return w, b, history

In [ ]:
# Test on linear dataset with scaling
X_tr_scratch, X_te_scratch, y_tr_scratch, y_te_scratch = train_test_split(
    X_lin, y_lin, test_size=0.25, random_state=42, stratify=y_lin
)

# Scale features (important!)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_scratch)
X_te_s = scaler.transform(X_te_scratch)

# Train from scratch
w, b, hist = train_linear_svm_from_scratch(X_tr_s, y_tr_scratch, C=1.0, lr=0.01, num_iters=2000, verbose=True)

# Helper function for prediction
def predict_linear_svm_from_scratch(X, w, b):
    """Predict using learned SVM parameters."""
    scores = X @ w + b
    predictions = (scores >= 0).astype(int)
    return predictions, scores

# Evaluate
y_pred_s, scores_s = predict_linear_svm_from_scratch(X_te_s, w, b)
accuracy = accuracy_score(y_te_scratch, y_pred_s)

print(f"\nFrom-scratch SVM accuracy: {accuracy:.4f}")

# Compare with scikit-learn
sklearn_model = train_linear_svm(X_tr_s, y_tr_scratch, C=1.0)
sklearn_acc = accuracy_score(y_te_scratch, sklearn_model.predict(X_te_s))
print(f"scikit-learn SVM accuracy: {sklearn_acc:.4f}")

##### **Expected Output**
```
Iteration 2000: Objective = X.XXXX

From-scratch SVM accuracy: 0.9XXX
scikit-learn SVM accuracy: 1.0000
```

In [ ]:
unittests.exercise_6c(train_linear_svm_from_scratch)

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(hist)
plt.title("Training Objective Over Time")
plt.xlabel("Iteration")
plt.ylabel("Objective Value")
plt.grid(True, alpha=0.3)
plt.show()

# Visualize decision boundary
class ScratchModel:
    """Wrapper for plotting compatibility."""
    def __init__(self, w, b):
        self.w = w
        self.b = b
    def predict(self, X):
        return (X @ self.w + self.b >= 0).astype(int)

plot_decision_boundary_2d(ScratchModel(w, b), X_tr_s, y_tr_scratch, 
                         title="From-Scratch Linear SVM (Scaled Data)")

## Congratulations!

You have successfully completed the Support Vector Machine assignment! You've built a comprehensive understanding of one of the most powerful and theoretically grounded machine learning algorithms.

### What You Accomplished:

1.  **Understood SVM Theory**: Learned about margin maximization, support vectors, and the kernel trick
2.  **Implemented Linear SVM**: Trained classifiers on linearly separable data and visualized decision boundaries
3.  **Demonstrated Feature Scaling**: Proved the critical importance of feature normalization for SVM performance
4.  **Applied RBF Kernel**: Used non-linear kernels to handle complex decision boundaries
5.  **Compared Kernels**: Evaluated linear, polynomial, and RBF kernels across multiple datasets
6.  **Multi-Class Strategies**: Implemented and compared One-vs-Rest and One-vs-One approaches
7.  **Built SVM from Scratch**: Implemented complete linear SVM using gradient descent
8.  **Tuned Hyperparameters**: Used grid search to find optimal C and gamma values

### Key Takeaways:

* **SVMs maximize the margin**: This geometric approach leads to robust decision boundaries
* **Support vectors are critical**: Only a small subset of training points defines the model
* **Feature scaling is essential**: SVMs are distance-based and highly sensitive to feature scales
* **Kernels enable non-linearity**: The kernel trick provides flexibility without explicit feature transformation
* **Hyperparameter tuning matters**: C and gamma significantly impact model performance
* **Multi-class strategies**: OvR and OvO extend binary classifiers to multi-class problems

### When to Use SVMs:

 **Good for:**
- Medium-sized datasets (hundreds to thousands of samples)
- High-dimensional spaces (text, genomics)
- Clear margin of separation exists
- Need for interpretable decision boundaries
- Limited training data available

 **Consider alternatives for:**
- Very large datasets (>100K samples) - training can be slow
- Highly noisy data with overlapping classes
- Need for probability estimates (requires calibration)
- Deep learning excels (images, speech, NLP with massive data)

**Great work! You now have a solid foundation in Support Vector Machines!**